# 02 - Decomposition Starter: RAG từng bước

Notebook này mô phỏng code mới, tách thành 2 phần như `ingest.py` và `query.py`:

**Ingest (idempotent)**
1. Import & cấu hình (`utils.py`)
2. Load tài liệu
3. Split tài liệu
4. Loại trùng & ghi vào Chroma (chạy nhiều lần không tạo bản ghi trùng)

**Query**
5. Khởi tạo LLM, retriever, prompt, chain
6. Decomposition: sinh sub-questions, trả lời từng câu, tổng hợp (starter)
7. Hàm `query()` và test

> Cần cài thêm `colorama` (dùng để in màu) và chạy `ingest` trước khi query.

In [ ]:
import hashlib
import warnings

from colorama import Fore
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")
load_dotenv()  # đọc OPENAI_API_KEY từ .env

True

## Bước 1: Cấu hình & hàm hỗ trợ (như `utils.py`)

In [27]:
PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "vnexpress"
URLS = [
    "https://vnexpress.net/thong-diep-5-tu-cua-elon-musk-khi-jensen-huang-phuc-chuc-5148278.html",
    "https://vnexpress.net/intel-sa-thai-mot-phan-nhan-su-it-toan-cau-5147238.html",
]


def make_id(doc) -> str:
    """ID xác định (deterministic) theo source + nội dung chunk."""
    key = f"{doc.metadata.get('source', '')}::{doc.page_content}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()


def get_vectorstore() -> Chroma:
    return Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=OpenAIEmbeddings(),
        persist_directory=PERSIST_DIR,
    )

## Bước 2: Load tài liệu

In [28]:
docs = []
for url in URLS:
    docs.extend(WebBaseLoader(url).load())

print(f"Số tài liệu: {len(docs)}")

Số tài liệu: 2


## Bước 3: Split tài liệu

In [29]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=50
)
splits = text_splitter.split_documents(docs)
for i, d in enumerate(splits):
    d.metadata["chunk_index"] = i

print(f"Số chunk: {len(splits)}")

Số chunk: 21


## Bước 4: Ingest idempotent vào Chroma

Loại trùng trong batch, bỏ qua chunk đã tồn tại, chỉ thêm chunk mới (như `ingest.py`).

In [31]:
# Loại trùng trong cùng batch
unique = {}
for d in splits:
    unique.setdefault(make_id(d), d)

vectorstore = get_vectorstore()
existing = set(vectorstore.get(ids=list(unique))["ids"])
new_ids = [i for i in unique if i not in existing]

if new_ids:
    vectorstore.add_documents([unique[i] for i in new_ids], ids=new_ids)

print(f"Chunks: {len(splits)} | unique: {len(unique)} | new: {len(new_ids)}")
print(f"Tổng bản ghi trong Chroma: {vectorstore._collection.count()}")

InternalError: Database error: error returned from database: (code: 1) no such table: acquire_write

## Bước 5: LLM, retriever, prompt, chain và hàm `query()` (như `query.py`)

In [32]:
llm = ChatOpenAI(model="gpt-4o-mini")
retriever = get_vectorstore().as_retriever(search_kwargs={"k": 4})

prompt = PromptTemplate.from_template(
    "Answer the question using only the context below.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)
chain = prompt | llm | StrOutputParser()


def query(question: str) -> str:
    context_docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in context_docs)
    return chain.invoke({"context": context, "question": question})

InternalError: Database error: error returned from database: (code: 14) unable to open database file

## Bước 6: Decomposition (như `query.py`)

Gồm 4 phần: (1) sinh sub-questions, (2) trả lời đệ quy từng sub-question, (3) RAG riêng từng sub-question, (4) tổng hợp câu trả lời cuối. Trong `query.py` phần 2-3 hiện mới là khung (`pass`); ở đây có bản chạy được để thực hành.

In [ ]:
# 1. Sinh sub-questions
template_decomposition = """You are a helpful assistant that generates multiple sub-questions related to an input question.
The goal is to break the input down into a set of sub-problems that can be answered in isolation.
Generate 3 search queries related to: {question}
Output (3 queries, one per line):"""
prompt_decomposition = PromptTemplate.from_template(template_decomposition)


def generate_sub_questions(question: str) -> list[str]:
    """Sinh sub-questions từ câu hỏi của người dùng."""
    chain_decomp = (
        prompt_decomposition | llm | StrOutputParser() | (lambda x: [q for q in x.split("\n") if q.strip()])
    )
    sub_questions = chain_decomp.invoke({"question": question})
    print(f"{Fore.GREEN}Sub-questions:\n" + "\n".join(sub_questions) + Fore.RESET)
    return sub_questions


# 2. Trả lời đệ quy: dùng Q+A trước đó làm ngữ cảnh bổ sung
template_qa = """Here is the question you need to answer:

\n --- \n {sub_question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question:

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {sub_question}
"""
prompt_qa = ChatPromptTemplate.from_template(template_qa)


def generate_qa_pairs(sub_questions: list[str]) -> str:
    q_a_pairs = ""
    for q in sub_questions:
        context = "\n\n".join(d.page_content for d in retriever.invoke(q))
        answer = (prompt_qa | llm | StrOutputParser()).invoke(
            {"sub_question": q, "q_a_pairs": q_a_pairs, "context": context}
        )
        q_a_pairs += f"\nQuestion: {q}\nAnswer: {answer}\n"
    return q_a_pairs


# 3. Trả lời từng sub-question độc lập (RAG prompt thay cho hub.pull)
prompt_rag = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. Use the following context "
    "to answer the question. If you don't know, say so. Keep it concise.\n"
    "Question: {question}\nContext: {context}\nAnswer:"
)


def retrieve_and_rag(prompt_rag, sub_questions: list[str]) -> list[str]:
    answers = []
    for q in sub_questions:
        context = "\n\n".join(d.page_content for d in retriever.invoke(q))
        answers.append((prompt_rag | llm | StrOutputParser()).invoke({"question": q, "context": context}))
    return answers


# 4. Tổng hợp câu trả lời cuối
template_final = """Here is a set of Q+A pairs:

{context}

to use these to synthesize an answer to the question: {question}
"""
prompt_final = ChatPromptTemplate.from_template(template_final)


def decompose_and_answer(question: str) -> str:
    sub_questions = generate_sub_questions(question)
    q_a_pairs = generate_qa_pairs(sub_questions)
    return (prompt_final | llm | StrOutputParser()).invoke({"context": q_a_pairs, "question": question})

## Bước 7: Test (câu hỏi mẫu xem `test_questions.md`)

In [ ]:
print(query("Intel đã làm gì?"))

print("\n=== Decomposition ===")
print(decompose_and_answer("Intel sa thải nhân sự nào và vì sao?"))

Intel sa thải một phần nhân sự IT toàn cầu.
